<a href="https://colab.research.google.com/github/NxrFesdac/bourbaki-nlp-avanzado/blob/main/modulo5-bbva/Multiagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langgraph langchain-core langchain-openai langchain-community langchain-tavily tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-co

In [ ]:
import os
import getpass
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langchain_community.tools.tavily_search import TavilySearchResults

# --- API Keys ---
os.environ["OPENAI_API_KEY"] = getpass.getpass()

os.environ["TAVILY_API_KEY"] = getpass.getpass()

··········
··········


In [ ]:
# --- 2. Definir el Estado ---
class EstadoAgente(TypedDict):
    messages: Annotated[list, operator.add]

llm = ChatOpenAI(model="gpt-5.4-nano")

# --- 3. Definir la Herramienta de Búsqueda Web Personalizada ---
@tool
def buscar_regulaciones_mx(query: str) -> str:
    """Busca en sitios gubernamentales y reguladores mexicanos leyes financieras y de fintech."""
    # Usando la importación de la comunidad como se solicitó
    busqueda_tavily = TavilySearchResults(max_results=2)
    consulta_estricta = f"{query} site:cnbv.gob.mx OR site:banxico.org.mx OR site:condusef.gob.mx OR site:diputados.gob.mx"
    return busqueda_tavily.invoke(consulta_estricta)

# --- 4. Definir el Nodo del Agente Creativo ---
def nodo_creativo(state: EstadoAgente):
    mensajes = state['messages']
    msg_sistema = SystemMessage(
        content="Eres el Director de Marketing de una nueva app Fintech mexicana llamada 'CriptoRápido'. "
                "Escribe un tuit promocional llamativo y atractivo. "
                "Si el Oficial de Cumplimiento te da retroalimentación, DEBES incluir cualquier aviso legal que te pida "
                "y eliminar por completo cualquier afirmación prohibida. Responde ÚNICAMENTE con el nuevo texto de marketing."
    )
    respuesta = llm.invoke([msg_sistema] + mensajes)
    return {"messages": [respuesta]}

# --- 5. Definir el Nodo del Agente Legal ---
instrucciones_legales = (
    "Eres un 'Oficial de Cumplimiento' para una Fintech mexicana. "
    "Revisa el texto de marketing específico proporcionado. DEBES usar tu herramienta de búsqueda para verificar las reglas actuales bajo la 'Ley Fintech' "
    "(específicamente sobre rendimientos garantizados, criptoactivos o divulgación de riesgos). "
    "Si el texto viola las regulaciones o carece de advertencias de riesgo obligatorias, DEBES responder con 'RECHAZADO: [Cita la regla, explica por qué y establece exactamente qué aviso legal debe agregarse]'. "
    "Si el texto es seguro, no hace falsas promesas e incluye los avisos necesarios, responde exactamente con 'APROBADO'."
)

agente_legal_ejecutable = create_agent(llm, tools=[buscar_regulaciones_mx], system_prompt=instrucciones_legales)

def nodo_legal(state: EstadoAgente):
    # Aislamiento de Contexto: Solo enviamos el último borrador al Agente Legal
    ultimo_borrador = state['messages'][-1].content
    solicitud_evaluacion = HumanMessage(
        content=f"Por favor, revisa el siguiente borrador de marketing para cumplimiento regulatorio:\n\n'{ultimo_borrador}'"
    )

    resultado = agente_legal_ejecutable.invoke({"messages": [solicitud_evaluacion]})
    return {"messages": [resultado['messages'][-1]]}

# --- 6. Definir el Router (Arista Condicional) ---
def debe_continuar(state: EstadoAgente):
    ultimo_mensaje = state['messages'][-1].content
    if "APROBADO" in ultimo_mensaje:
        return "fin"
    else:
        return "continuar"

# --- 7. Construir el Grafo ---
flujo_trabajo = StateGraph(EstadoAgente)

flujo_trabajo.add_node("creativo", nodo_creativo)
flujo_trabajo.add_node("legal", nodo_legal)

flujo_trabajo.set_entry_point("creativo")
flujo_trabajo.add_edge("creativo", "legal")

flujo_trabajo.add_conditional_edges(
    "legal",
    debe_continuar,
    {"continuar": "creativo", "fin": END}
)

app = flujo_trabajo.compile()


# Comenzando con la petición "ilegal"
entrada_inicial = {
    "messages": [HumanMessage(content="¡Escribe un tuit prometiendo rendimientos del 100% garantizados y sin riesgo en inversiones de Bitcoin con nuestra app!")]
}

for salida in app.stream(entrada_inicial, {"recursion_limit": 10}):
    for nombre_nodo, estado_nodo in salida.items():
        print(f"\n--- AGENTE {nombre_nodo.upper()} ---")
        print(estado_nodo["messages"][0].content)

Starting the Multi-Agent Simulation...


--- CREATIVE AGENT ---
¡Convierte tu curiosidad en acción con **CriptoRápido**! 📲⚡ Compra y vende Bitcoin de forma rápida y práctica desde tu celular.  

Invertir en criptoactivos implica **riesgos**; el valor puede subir o bajar y podrías perder parte o la totalidad de tu inversión.

--- LEGAL AGENT ---
REJECTED: La ley aplicable (Ley para Regular las Instituciones de Tecnología Financiera – LRTIF) exige que las ITF incluyan en su publicidad **la “divulgación de riesgos y responsabilidades… en un lenguaje sencillo y claro”** para la adecuada toma de decisión (y que dicha divulgación incluya **advertencias relativas a la utilización** de la interfaz/página/medio digital correspondiente). En tu copy solo se incluye una advertencia general (“puedes perder…”) pero **no incorpora el conjunto mínimo de advertencias/elementos de riesgo** típicos para criptoactivos (p. ej., volatilidad, que no es moneda de curso legal ni está respaldado por el Gobierno